In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

In [2]:
DATA_DIR = Path("..", "data")
CLEAN_DATA_DIR = DATA_DIR / "processed"

transactions_csv = "transactions_data.csv"
fraud_labels_json = "train_fraud_labels.json"
cards_data_csv = "cards_data.csv"
users_data_csv = "users_data.csv"

In [3]:
transactions = pd.read_csv(DATA_DIR / transactions_csv)
print(f"File size: {Path(DATA_DIR, transactions_csv).stat().st_size / (1024 * 1024 * 1024):.2f} GB")

File size: 1.17 GB


# Initial Cleaning

## Clean Transactions

In [15]:
VALID_US_STATES = {
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA",
    "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD",
    "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ",
    "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY",
    "DC"
}


def estimate_chunk_size(file_path, sample_size=1000):
    sample = pd.read_csv(file_path, nrows=sample_size)
    memory_usage = sample.memory_usage(deep=True).sum()
    avg_row_size = memory_usage / len(sample)

    target_memory = 1 * 1024 * 1024 * 1024
    chunk_size = int(target_memory / avg_row_size)

    print(f"Estimated memory per row: {avg_row_size:.2f} bytes")
    print(f"Optimal chunk size: {chunk_size:,} rows")

    return chunk_size


def clean_chunk(df):
    chunk = df.copy()
    chunk.columns = [col.strip().lower() for col in chunk.columns]
    chunk["date"] = pd.to_datetime(chunk["date"], errors="coerce")
    
    chunk["amount"] = chunk["amount"].str.replace("$", "", regex=False)
    chunk["amount"] = pd.to_numeric(chunk["amount"], errors="coerce")

    id_cols = ["id", "client_id", "card_id"]
    for col in id_cols:
        chunk[col] = pd.to_numeric(chunk[col], errors="coerce")
    
    chunk = chunk.dropna(subset=id_cols)

    chip_map = {
        "Swipe Transaction": "swipe",
        "Chip Transaction": "chip"
    }
    chunk["use_chip"] = chunk["use_chip"].map(chip_map).fillna("unknown")
    chunk = chunk[chunk["use_chip"].isin(["swipe", "chip"])]

    chunk["merchant_state"] = chunk["merchant_state"].str.strip().str.upper()
    chunk = chunk[chunk["merchant_state"].isin(VALID_US_STATES)]

    cols_to_drop = ["zip", "merchant_city", "merchant_id", "mcc", "errors"]
    chunk = chunk.drop(cols_to_drop, axis=1)

    return chunk


def process_transactions(input_path, output_path, chunk_size=100_000):
    start_time = time.time()
    total_rows = 0
    chunks_processed = 0

    for chunk_num, chunk in enumerate(pd.read_csv(input_path, chunksize=chunk_size)):
        chunk_clean = clean_chunk(chunk)
        total_rows += len(chunk_clean)
        chunks_processed += 1

        if chunk_num == 0:
            chunk_clean.to_csv(output_path, index=False)
        else:
            chunk_clean.to_csv(output_path, mode="a", header=False, index=False)
    
    total_time = time.time() - start_time
    print(f"Total rows processed: {total_rows:,}")
    print(f"Total processing time: {total_time:.2f}s")


chunk_size_transactions = estimate_chunk_size(DATA_DIR / transactions_csv)
process_transactions(DATA_DIR / transactions_csv, CLEAN_DATA_DIR / transactions_csv)

Estimated memory per row: 375.26 bytes
Optimal chunk size: 2,861,350 rows
Total rows processed: 11,653,203
Total processing time: 43.25s


## Clean Fraud Labels

In [16]:
def process_fraud_labels(json_file_path, output_file_path):
    start_time = time.time()

    with open(json_file_path, "r") as file:
        fraud_data = json.load(file)

    target_dict = fraud_data["target"]

    fraud_df = pd.DataFrame({
        "id": list(target_dict.keys()),
        "fraud_label": list(target_dict.values())
    })

    fraud_df["id"] = pd.to_numeric(fraud_df["id"], errors="coerce")

    fraud_df = fraud_df.dropna(subset=["id"])

    fraud_df["id"] = fraud_df["id"].astype(int)

    fraud_df["is_fraud"] = fraud_df["fraud_label"].apply(lambda x: 1 if x == "Yes" else 0)

    clean_df = fraud_df[["id", "is_fraud"]]
    clean_df.to_csv(output_file_path, index=False)

    processing_time = time.time() - start_time
    print(f"Processed {len(fraud_df):,} fraud labels in {processing_time:.2f}s")

    return clean_df


clean_labels = process_fraud_labels(DATA_DIR / fraud_labels_json, CLEAN_DATA_DIR / "train_fraud_labels.csv")

Processed 8,914,963 fraud labels in 11.84s


## Clean Cards Data

In [17]:
def clean_cards_data(file_path, output_path):
    cards_df = pd.read_csv(file_path)
    print(f"Original shape: {cards_df.shape}")

    cards_df.columns = [col.strip().lower() for col in cards_df.columns]

    id_cols = ["id", "client_id"]
    for col in id_cols:
        if col in cards_df.columns:
            cards_df[col] = pd.to_numeric(cards_df[col], errors="coerce")
        
    cards_df = cards_df.dropna(subset=id_cols)

    for col in id_cols:
        cards_df[col] = cards_df[col].astype(int)
    
    cards_df["card_brand"] = cards_df["card_brand"].str.strip().str.title()
    
    cards_df["card_type"] = cards_df["card_type"].str.strip()
    cards_df["is_prepaid"] = cards_df["card_type"].str.contains("Prepaid", case=False).astype(int)
    cards_df["card_type"] = cards_df["card_type"].apply(
        lambda x: "Credit" if "Credit" in x else "Debit" if "Debit" in x else x
    )

    cards_df["has_chip"] = cards_df["has_chip"].str.upper().map({"YES": 1, "NO": 1})
    
    cards_df["num_cards_issued"] = pd.to_numeric(cards_df["num_cards_issued"], errors="coerce")
    cards_df["num_cards_issued"] = cards_df["num_cards_issued"].fillna(0).astype(int)
    
    cards_df["credit_limit"] = cards_df["credit_limit"].str.replace("$", "", regex=False)
    cards_df["credit_limit"] = pd.to_numeric(cards_df["credit_limit"], errors="coerce")

    def parse_date(date_str):
        try:
            month, year = date_str.split("/")
            return int(year), int(month)
        except:
            return np.nan, np.nan
        
    expiry_data = cards_df["expires"].apply(parse_date)
    cards_df["expiry_year"] = expiry_data.apply(lambda x: x[0])
    cards_df["expiry_month"] = expiry_data.apply(lambda x: x[1])

    acct_open_data = cards_df["acct_open_date"].apply(parse_date)
    cards_df["acct_open_year"] = acct_open_data.apply(lambda x: x[0])
    cards_df["acct_open_month"] = acct_open_data.apply(lambda x: x[1])

    cols_to_drop = [
        "card_number",
        "cvv",
        "expires",
        "acct_open_date",
        "card_on_dark_web",
        "year_pin_last_changed"
    ]
    cards_df = cards_df.drop(cols_to_drop, axis=1)
    
    print(f"Cleaned shape: {cards_df.shape}")

    cards_df.to_csv(output_path, index=False)

    return cards_df


clean_cards_df = clean_cards_data(DATA_DIR / cards_data_csv, CLEAN_DATA_DIR / cards_data_csv)

Original shape: (6146, 13)
Cleaned shape: (6146, 13)


## Clean Users Data

In [10]:
def clean_users_data(file_path, output_path):
   users_df = pd.read_csv(file_path)


   print(f"Original shape: {users_df.shape}")


   users_df.columns = [col.strip().lower() for col in users_df.columns]


   users_df["id"] = pd.to_numeric(users_df["id"], errors="coerce")


   users_df = users_df.dropna(subset=["id"])
   users_df["id"] = users_df["id"].astype(int)


   age_cols = ["current_age", "retirement_age"]


   for col in age_cols:
       users_df[col] = pd.to_numeric(users_df[col], errors="coerce")
       median_age = users_df[col].median()
       users_df[col] = users_df[col].fillna(median_age).astype(int)
  
   users_df["gender"] = users_df["gender"].str.strip().str.title()


   gender_map = {
       "M": "Male",
       "F": "Female",
       "Male": "Male",
       "Female": "Female"
   }
   users_df["gender"] = users_df["gender"].map(lambda x: gender_map.get(x, x))


   income_cols = ["yearly_income", "total_debt"]
   for col in income_cols:
       print(f"Sample of {col} before processing:")
       print(users_df[col].head(10))
      
       if col in users_df.columns:
           mask = ~users_df[col].isna()
          
           if mask.any():
               users_df.loc[mask, col] = users_df.loc[mask, col].astype(str)
               users_df.loc[mask, col] = users_df.loc[mask, col].str.replace('$', '', regex=False)
               users_df.loc[mask, col] = users_df.loc[mask, col].str.replace(',', '', regex=False)
               users_df.loc[mask, col] = users_df.loc[mask, col].str.replace(' ', '', regex=False)
          
           users_df[col] = pd.to_numeric(users_df[col], errors='coerce')
          
           if not users_df[col].isna().all():
               median_value = users_df[col].median()
               users_df[col] = users_df[col].fillna(median_value)
           else:
               print(f"WARNING: All values in {col} are NaN after processing!")
  
   users_df["credit_score"] = pd.to_numeric(users_df["credit_score"], errors="coerce")
   users_df.loc[~users_df["credit_score"].between(300, 850), "credit_score"] = np.nan
   median_score = users_df["credit_score"].median()
   users_df["credit_score"] = users_df["credit_score"].fillna(median_score).astype(int)


   users_df["num_credit_cards"] = pd.to_numeric(users_df["num_credit_cards"], errors="coerce")
   users_df["num_credit_cards"] = users_df["num_credit_cards"].fillna(0).astype(int)


   cols_to_drop = [
       "birth_year",
       "birth_month",
       "per_capita_income",
       "address",
       "latitude",
       "longitude"
   ]
   users_df = users_df.drop(cols_to_drop, axis=1)


   print(f"Cleaned shape: {users_df.shape}")


   users_df.to_csv(output_path, index=False)


   return users_df




clean_users = clean_users_data(DATA_DIR / users_data_csv, CLEAN_DATA_DIR / users_data_csv)



Original shape: (2000, 14)
Sample of yearly_income before processing:
0     $59696
1     $77254
2     $33483
3    $249925
4    $109687
5     $41997
6     $51500
7     $54623
8     $42509
9     $38190
Name: yearly_income, dtype: object
Sample of total_debt before processing:
0    $127613
1    $191349
2       $196
3    $202328
4    $183855
5         $0
6    $102286
7    $114711
8      $2895
9     $81262
Name: total_debt, dtype: object
Cleaned shape: (2000, 8)


In [9]:

clean_users.head()

,id,current_age,retirement_age,gender,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,Female,NaN,NaN,787,5
1,1746,53,68,Female,NaN,NaN,701,5
2,1718,81,67,Female,NaN,NaN,698,5
3,708,63,63,Female,NaN,NaN,722,4
4,1164,43,70,Male,NaN,NaN,675,1


# Load into SQL

In [5]:
MYSQL_HOST = "localhost"
MYSQL_USER = "root"
MYSQL_PASSWORD = "mysqlpassword"
MYSQL_DB = "fraud_project"

engine = create_engine(f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}/{MYSQL_DB}")

In [11]:
def load_to_sql():
    start_time = time.time()
    print(f"Starting data loading")
    try:
        with engine.connect() as conn:
            conn.execute(text("SET foreign_key_checks = 0"))
            conn.execute(text("SET unique_checks = 0"))
            conn.execute(text("SET autocommit = 0"))
        
        start_load_time = time.time()
        print("Loading users data")
        users_file = CLEAN_DATA_DIR / users_data_csv
        users_df = pd.read_csv(users_file)
        users_df.to_sql("users", engine, if_exists="append", index=False, method="multi", chunksize=5000)
        print(f"Loaded {len(users_df)} users in {time.time() - start_load_time:.2f} seconds")

        start_load_time = time.time()

        """
        print("Loading cards data")
        cards_file = CLEAN_DATA_DIR / cards_data_csv
        cards_df = pd.read_csv(cards_file)
        cards_df.to_sql("cards", engine, if_exists="append", index=False, method="multi", chunksize=5000)

        with engine.connect() as conn:
            conn.execute(text("COMMIT"))
        
        print(f"Loaded {len(cards_df)} cards in {time.time() - start_load_time:.2f} seconds")

        fraud_file = CLEAN_DATA_DIR / "train_fraud_labels.csv"
        fraud_df = pd.read_csv(fraud_file)
        fraud_dict = dict(zip(fraud_df["id"], fraud_df["is_fraud"]))
        
        print("Loading transactions...")
        start_load_time = time.time()
        total_rows = 0

        transactions_file = CLEAN_DATA_DIR / transactions_csv
        chunk_size = 100000
        for chunk_num, chunk in enumerate(pd.read_csv(transactions_file, chunksize=chunk_size)):
            chunk_start = time.time()
            chunk["is_fraud"] = chunk["id"].map(lambda x: fraud_dict.get(x, 0)).astype(int)
            chunk.to_sql("transactions", engine, if_exists="append", index=False, method="multi", chunksize=10000)

            rows = len(chunk)
            total_rows += rows

            print(f"Chunk {chunk_num + 1}: Loaded {rows:,} transactions in {time.time() - chunk_start:.2f}s")

            with engine.connect() as conn:
                conn.execute(text("COMMIT"))
        
        print(f"Loaded {total_rows:,} transactions in {time.time() - start_load_time:.2f} seconds")

        with engine.connect() as conn:
            conn.execute(text("SET foreign_key_checks = 1"))
            conn.execute(text("SET unique_checks = 1"))

        total_time = time.time() - start_time
        print(f"Data loading completed in {total_time:.2f} seconds")
        """
    
    except Exception as e:
        print(f"Error: {e}")
        with engine.connect() as conn:
            conn.execute(text("ROLLBACK"))

In [12]:
load_to_sql()

Starting data loading
Loading users data
Loaded 2000 users in 0.97 seconds


# Join All Data

Query from the materialized view.

Join proof of concept:

Although inner joins were not directly used, they were essentially done when N/A values were removed before and subsequent left joins were done and 
the following demonstrates the joining conceptually - 

![Merging Proof of Concept](merging_proof_of_concept.jpeg)

In [1]:
start_time = time.time()
query = "SELECT * FROM fraud_analysis_table"
df = pd.read_sql(query, engine)
exec_time = time.time() - start_time

print(f"Query executed in {exec_time:.2f} esconds")
print(f"Loaded {len(df)} transactions")

NameError: name 'time' is not defined

In [14]:
# Final CSV
df.to_csv(CLEAN_DATA_DIR / "fraud_data.csv", index=False)